In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1. Synthesize daily website traffic data with weekly seasonality (s=7)
np.random.seed(42)
n_days = 120
dates = pd.date_range(start='2026-01-01', periods=n_days, freq='D')

# Create an exogenous variable: Ad Campaign Spend (Random fluctuations)
ad_spend = np.random.uniform(10, 50, size=n_days)

# Create the target variable: Traffic depends on Ad Spend + Weekly Cyclical Patterns
base_traffic = 100 + (ad_spend * 1.5)
weekly_pattern = np.array([20, 40, 10, -5, -15, 30, 50]) # High weekend traffic
seasonal_traffic = np.array([weekly_pattern[d.weekday()] for d in dates])
noise = np.random.normal(0, 5, size=n_days)

traffic = base_traffic + seasonal_traffic + noise

# Build clean DataFrames
df_target = pd.Series(traffic, index=dates)
df_exog = pd.DataFrame({'Ad_Spend': ad_spend}, index=dates)

# Split into Training (110 days) and Testing (10 days out)
train_y, test_y = df_target.iloc[:-10], df_target.iloc[-10:]
train_X, test_X = df_exog.iloc[:-10], df_exog.iloc[-10:]

In [2]:
# 2. Initialize and fit the SARIMAX model
# order=(1,1,1) handles baseline trends
# seasonal_order=(1,1,1,7) handles weekly seasonal patterns
model = SARIMAX(
    train_y, 
    exog=train_X, 
    order=(1, 1, 1), 
    seasonal_order=(1, 1, 1, 7)
)
results = model.fit(disp=False)

print("--- SARIMAX Parameter Evaluation ---")
print(results.summary().tables[1])

--- SARIMAX Parameter Evaluation ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Ad_Spend       1.5002      0.036     41.188      0.000       1.429       1.572
ar.L1         -0.0809      0.111     -0.729      0.466      -0.298       0.137
ma.L1         -0.9996      6.646     -0.150      0.880     -14.026      12.027
ar.S.L7        0.1331      0.133      1.004      0.315      -0.127       0.393
ma.S.L7       -0.9991     17.277     -0.058      0.954     -34.862      32.863
sigma2        22.2598    441.323      0.050      0.960    -842.717     887.237


In [3]:
# 3. Forecast future steps
# CRITICAL REQUIREMENT: To forecast with exogenous variables, you MUST provide 
# the future planned values of those exogenous variables (test_X)
forecast = results.forecast(steps=10, exog=test_X)

print("\n--- Production Traffic Forecast vs Actual ---")
forecast_comparison = pd.DataFrame({'Actual': test_y, 'Forecast': forecast})
print(forecast_comparison.round(2))


--- Production Traffic Forecast vs Actual ---
            Actual  Forecast
2026-04-21  177.80    172.98
2026-04-22  139.94    133.94
2026-04-23  158.89    167.56
2026-04-24  143.80    149.03
2026-04-25  185.58    182.81
2026-04-26  219.86    217.17
2026-04-27  185.80    184.44
2026-04-28  185.46    166.83
2026-04-29  181.41    178.37
2026-04-30  148.04    144.33
